# MCP Protocol Fundamentals

The Model Context Protocol (MCP, Anthropic 2024) is an open standard for connecting LLMs to external data sources and tools. It replaced the ad-hoc plugin and function-calling approaches with a structured, transport-agnostic protocol.

## Before MCP: The Plugin Fragmentation Problem

ChatGPT plugins (OpenAI, 2023) were the first large-scale attempt at standardized LLM tool integration. The problems:
- Each plugin required a custom OpenAPI spec and hosting infrastructure
- No standard for authentication, capability discovery, or error handling
- Plugin quality was inconsistent; the model could not distinguish reliable from unreliable plugins
- Plugins were tightly coupled to ChatGPT's specific function-calling format

OpenAI's function calling v1 (2023) improved the model-side interface but left the server-side implementation completely unspecified. Every tool integration required custom code.

MCP solved this by defining a complete protocol: how servers advertise capabilities, how clients discover and invoke them, what error codes mean, and how to handle authentication.

## MCP Architecture

MCP has three components:

**MCP Host**: the application that contains the LLM (e.g., Claude Desktop, your custom agent). The host orchestrates everything.

**MCP Client**: a connection to one MCP server. The host manages one client per connected server.

**MCP Server**: a lightweight service that exposes capabilities via the MCP protocol. Servers expose three primitive types:
- **Resources**: read-only data (files, database records, API responses)
- **Tools**: executable functions with side effects
- **Prompts**: reusable prompt templates

The transport layer is pluggable — MCP works over stdio (for local servers) or HTTP+SSE (for remote servers).

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable, Optional
import json

@dataclass
class MCPResource:
    uri: str
    name: str
    description: str
    mime_type: str = 'text/plain'
    content: str = ''

@dataclass
class MCPTool:
    name: str
    description: str
    input_schema: dict
    fn: Callable

    def to_spec(self) -> dict:
        return {'name': self.name, 'description': self.description, 'inputSchema': self.input_schema}

@dataclass
class MCPPrompt:
    name: str
    description: str
    arguments: list
    template: str

    def render(self, args: dict) -> str:
        result = self.template
        for k, v in args.items():
            result = result.replace('{'+k+'}', str(v))
        return result

class MCPServer:
    def __init__(self, name: str, version: str = '1.0.0'):
        self.name = name
        self.version = version
        self.resources: dict = {}
        self.tools: dict = {}
        self.prompts: dict = {}

    def add_resource(self, resource: MCPResource): self.resources[resource.uri] = resource
    def add_tool(self, tool: MCPTool): self.tools[tool.name] = tool
    def add_prompt(self, prompt: MCPPrompt): self.prompts[prompt.name] = prompt

    def handle(self, method: str, params: dict) -> dict:
        if method == 'initialize':
            return {'protocolVersion': '2024-11-05', 'serverInfo': {'name': self.name, 'version': self.version},
                    'capabilities': {'resources': {}, 'tools': {}, 'prompts': {}}}
        if method == 'tools/list':
            return {'tools': [t.to_spec() for t in self.tools.values()]}
        if method == 'tools/call':
            name = params.get('name'); args = params.get('arguments', {})
            if name not in self.tools:
                return {'error': {'code': -32601, 'message': f'Tool not found: {name}'}}
            try:
                result = self.tools[name].fn(**args)
                return {'content': [{'type': 'text', 'text': str(result)}]}
            except Exception as e:
                return {'error': {'code': -32603, 'message': str(e)}}
        if method == 'resources/list':
            return {'resources': [{'uri': r.uri, 'name': r.name, 'description': r.description}
                                   for r in self.resources.values()]}
        if method == 'resources/read':
            uri = params.get('uri')
            if uri not in self.resources:
                return {'error': {'code': -32601, 'message': f'Resource not found: {uri}'}}
            r = self.resources[uri]
            return {'contents': [{'uri': uri, 'mimeType': r.mime_type, 'text': r.content}]}
        return {'error': {'code': -32601, 'message': f'Unknown method: {method}'}}

# Build a demo MCP server
server = MCPServer('demo-server', '1.0.0')
server.add_tool(MCPTool(
    name='get_time',
    description='Returns the current time in ISO format',
    input_schema={'type': 'object', 'properties': {}, 'required': []},
    fn=lambda: '2024-04-13T10:30:00Z'
))
server.add_tool(MCPTool(
    name='calculate',
    description='Evaluate a mathematical expression',
    input_schema={'type': 'object', 'properties': {'expression': {'type': 'string'}}, 'required': ['expression']},
    fn=lambda expression: eval(expression, {'__builtins__': {}}, {})
))
server.add_resource(MCPResource('file:///readme.txt', 'README', 'Project readme', 'text/plain', 'Welcome to the demo MCP server.'))

# Simulate MCP protocol exchange
print('=== MCP Protocol Simulation ===')
print('1. Initialize:')
print(json.dumps(server.handle('initialize', {}), indent=2)[:200])
print('\n2. List tools:')
print(json.dumps(server.handle('tools/list', {}), indent=2))
print('\n3. Call calculate tool:')
print(json.dumps(server.handle('tools/call', {'name': 'calculate', 'arguments': {'expression': '15 * 0.20'}})))
print('\n4. Read resource:')
print(json.dumps(server.handle('resources/read', {'uri': 'file:///readme.txt'})))

## Why MCP Won

MCP succeeded where previous approaches failed because it solved the right problem at the right layer:

**Separation of concerns**: the protocol specifies *what* servers must expose (capability discovery, resource/tool/prompt primitives) but not *how* they implement it. A filesystem server, a database server, and an API server can all speak MCP.

**Composability**: a single host can connect to multiple MCP servers simultaneously. Users can mix-and-match servers from different providers.

**Local-first**: stdio transport means sensitive tools (file system, local database) never require network exposure.

**Adoption**: Anthropic open-sourced the spec and SDKs, and built-in Claude Desktop support drove immediate adoption. By early 2025, hundreds of MCP servers existed for common tools (GitHub, Postgres, Slack, Jira).